# US Airports Geography and Weather Anomaly Analysis

This notebook evaluates spatial and temporal meteorological attributes across the United States using the `nycflights13` dataset. By investigating spatial coordinates and regional weather observations, this analysis identifies extreme geographical limits (northernmost and easternmost US airports) and pinpoints localized weather events, specifically isolating anomalous environmental data and documenting research assumptions and corrections.

In [ ]:
import pandas as pd

# Load datasets
airports = pd.read_csv("https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/airports.csv")
weather = pd.read_csv("https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/weather.csv")

# Display basic information
print("Airports columns:", airports.columns.tolist())
print("Weather columns:", weather.columns.tolist())
display(airports.head())
display(weather.head())

Airports columns: ['faa', 'name', 'lat', 'lon', 'alt', 'tz', 'dst', 'tzone']
Weather columns: ['origin', 'year', 'month', 'day', 'hour', 'temp', 'dewp', 'humid', 'wind_dir', 'wind_speed', 'wind_gust', 'precip', 'pressure', 'visib', 'time_hour']


,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


,origin,year,month,day,hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour
0,EWR,2013,1,1,1,39.02,26.06,59.37,270.0,10.35702,NaN,0.0,1012.0,10.0,2013-01-01T06:00:00Z
1,EWR,2013,1,1,2,39.02,26.96,61.63,250.0,8.05546,NaN,0.0,1012.3,10.0,2013-01-01T07:00:00Z
2,EWR,2013,1,1,3,39.02,28.04,64.43,240.0,11.50780,NaN,0.0,1012.5,10.0,2013-01-01T08:00:00Z
3,EWR,2013,1,1,4,39.92,28.04,62.21,250.0,12.65858,NaN,0.0,1012.2,10.0,2013-01-01T09:00:00Z
4,EWR,2013,1,1,5,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z


In [ ]:
# 1. Northernmost airport in the US
print("--- Top 5 Northernmost Airports ---")
northernmost = airports.sort_values(by="lat", ascending=False).head(5)
display(northernmost[["faa", "name", "lat", "lon"]])

# 2. Easternmost airport in the US
# Note: Longitude values in the US are generally negative (Western hemisphere).
# The closer to 0 or positive (if crossing the 180th meridian/International Date Line), the more eastern it is.
print("\n--- Top 5 Easternmost Airports ---")
easternmost = airports.sort_values(by="lon", ascending=False).head(5)
display(easternmost[["faa", "name", "lat", "lon"]])

# 3. Windiest NYC area airport on February 12, 2013
print("\n--- Weather on Feb 12, 2013 ---")
feb12_weather = weather[(weather["year"] == 2013) & (weather["month"] == 2012) & (weather["day"] == 12)]
feb12_weather = weather[(weather["year"] == 2013) & (weather["month"] == 2) & (weather["day"] == 12)]

# Let's inspect maximum wind speeds and gusts
windiest_by_speed = feb12_weather.sort_values(by="wind_speed", ascending=False).head(5)
windiest_by_gust = feb12_weather.sort_values(by="wind_gust", ascending=False).head(5)

print("Top wind speed instances:")
display(windiest_by_speed[["origin", "hour", "wind_speed", "wind_gust"]])
print("Top wind gust instances:")
display(windiest_by_gust[["origin", "hour", "wind_speed", "wind_gust"]])

--- Top 5 Northernmost Airports ---


,faa,name,lat,lon
417,EEN,Dillant Hopkins Airport,72.270833,42.898333
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003
110,AIN,Wainwright Airport,70.638056,-159.994722
708,K03,Wainwright As,70.613378,-159.860350
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000



--- Top 5 Easternmost Airports ---


,faa,name,lat,lon
1290,SYA,Eareckson As,52.712275,174.113620
942,MYF,Montgomery Field,32.475900,117.759000
396,DVT,Deer Valley Municipal Airport,33.411700,112.457000
417,EEN,Dillant Hopkins Airport,72.270833,42.898333
444,EPM,Eastport Municipal Airport,44.910111,-67.012694



--- Weather on Feb 12, 2013 ---
Top wind speed instances:


,origin,hour,wind_speed,wind_gust
1009,EWR,3,1048.36058,NaN
18417,LGA,2,23.01560,31.07106
1018,EWR,12,21.86482,31.07106
18428,LGA,13,21.86482,25.31716
9717,JFK,8,20.71404,27.61872


Top wind gust instances:


,origin,hour,wind_speed,wind_gust
18417,LGA,2,23.01560,31.07106
1019,EWR,13,13.80936,31.07106
1018,EWR,12,21.86482,31.07106
18421,LGA,6,13.80936,28.76950
18418,LGA,3,16.11092,28.76950


In [ ]:
# Let's inspect the exact values for wind speed on Feb 12, 2013 to find any outliers
print("--- Wind Speed Outlier Check (Feb 12, 2013) ---")
display(feb12_weather.sort_values(by="wind_speed", ascending=False)[["origin", "hour", "wind_speed", "wind_gust"]].head(10))

# Let's filter out physically impossible wind speeds (e.g. > 100 mph, since hurricane strength is about 74+ mph, and the windiest recorded non-tornadic gust on Earth is around 253 mph)
valid_weather = feb12_weather[feb12_weather["wind_speed"] < 100]
print("\n--- Windiest after removing sensor errors/outliers ---")
display(valid_weather.sort_values(by="wind_speed", ascending=False)[["origin", "hour", "wind_speed", "wind_gust"]].head(5))

--- Wind Speed Outlier Check (Feb 12, 2013) ---


,origin,hour,wind_speed,wind_gust
1009,EWR,3,1048.36058,NaN
18417,LGA,2,23.01560,31.07106
1018,EWR,12,21.86482,31.07106
18428,LGA,13,21.86482,25.31716
9717,JFK,8,20.71404,27.61872
1008,EWR,2,20.71404,25.31716
18429,LGA,14,20.71404,25.31716
18416,LGA,1,19.56326,28.76950
9716,JFK,7,19.56326,NaN
18427,LGA,12,19.56326,26.46794



--- Windiest after removing sensor errors/outliers ---


,origin,hour,wind_speed,wind_gust
18417,LGA,2,23.01560,31.07106
1018,EWR,12,21.86482,31.07106
18428,LGA,13,21.86482,25.31716
9717,JFK,8,20.71404,27.61872
1008,EWR,2,20.71404,25.31716


## Analysis and Conclusions

### 1. Northernmost US Airport
Based on a simple latitude query of the dataset, **Dillant Hopkins Airport (EEN)** initially appears to be the northernmost airport with a latitude of **72.27° N**. However, this is an error in the source data:
* **Data Correction/Validation:** Dillant Hopkins Airport is actually located in **Keene, New Hampshire** (approx. 42.9° N, 72.3° W). The dataset has swapped its latitude and longitude values.
* **Final Answer:** After resolving this swap, **Wiley Post Will Rogers Memorial Airport (BRW)** in Utqiaġvik (Barrow), Alaska, at **71.29° N** is the true northernmost airport in the United States.

### 2. Easternmost US Airport
When analyzing the easternmost airport in the US using longitude, we must account for coordinate systems and the International Date Line:
* **Geographic Easternmost:** **Eareckson Air Station (SYA)** on the island of Shemya in the Western Aleutian Islands, Alaska, has a longitude of **174.11° E**. Because it crosses the 180th meridian into the Eastern Hemisphere, it is geographically the easternmost US airport.
* **Contiguous/North American Landmass Easternmost:** If we exclude airports that cross into the Eastern Hemisphere, **Eastport Municipal Airport (EPM)** in Maine is the easternmost US airport at **67.01° W** longitude.
* **Data Anomalies:** Note that `EEN` (coordinate swap), `MYF` (San Diego), and `DVT` (Phoenix) also appeared at the top of the uncleaned longitude sort due to missing negative signs or coordinate typos.

### 3. Windiest NYC Area Airport on February 12th, 2013
Our initial weather dataset query produced a massive outlier:
* **Data Outlier:** At 3:00 AM on Feb 12th, 2013, **EWR (Newark)** logged a wind speed of **1048.36 mph**. This is physically impossible (faster than the speed of sound) and represents a sensor malfunction.
* **Final Answer:** After filtering out this outlier, **LGA (LaGuardia Airport)** was the windiest airport on February 12th, 2013, peaking at a wind speed of **23.02 mph** with gusts up to **31.07 mph** at 2:00 AM.